In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, MultiHeadAttention, LayerNormalization, Conv1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import categorical_crossentropy
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
# Constants
VIDEO_FOLDER1 = 'data/videos'
VIDEO_FOLDER2  = '../video-p-reaction-joketype-modular/data/videos'
DATA_FILE = '/Users/jeremiah/video-p-reaction-joketype-modular/data/excel.xlsx'
N_FRAMES = 5
FRAME_HEIGHT = 224
FRAME_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 5
MC_ITERATIONS = 4
LSTM_UNITS = 128

In [8]:
# Load and preprocess data

data = pd.read_excel(DATA_FILE)
X = data['VideoID']
y_joke = data['JokeType']
y_reaction = data['HowFunny']

le_joke = LabelEncoder()
le_reaction = LabelEncoder()
y_joke_encoded = le_joke.fit_transform(y_joke)
y_reaction_encoded = le_reaction.fit_transform(y_reaction)

X_train, X_test, y_train_joke, y_test_joke, y_train_reaction, y_test_reaction = train_test_split(
    X, y_joke_encoded, y_reaction_encoded, test_size=0.2, random_state=42
)

In [9]:
# Feature extraction
base_model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

def extract_features(video_files):
    features = []
    valid_indices = []
    for i, video_file in enumerate(video_files):
        video_path = os.path.join(VIDEO_FOLDER1, video_file)
        if not os.path.exists(video_path):
            print(f"Video file not found: {video_path}")
            continue
        
        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_features = []
        
        if frame_count == 0:
            print(f"No frames found in video: {video_path}")
            continue
        
        for j in range(N_FRAMES):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * j / N_FRAMES))
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
                frame = preprocess_input(frame)
                frame_features.append(feature_extractor.predict(np.expand_dims(frame, axis=0)).flatten())
            else:
                print(f"Failed to read frame {j} from video: {video_path}")
                break
        
        cap.release()
        
        if len(frame_features) == N_FRAMES:
            features.append(np.array(frame_features))
            valid_indices.append(i)
        else:
            print(f"Skipping video due to insufficient frames: {video_path}")
    
    if len(features) == 0:
        raise ValueError("No valid features extracted from any videos")
    
    return np.array(features), valid_indices

# Extract features and get valid indices
X_train_features, train_valid_indices = extract_features(X_train)
X_test_features, test_valid_indices = extract_features(X_test)

# Update labels to match valid samples
y_train_joke = y_train_joke[train_valid_indices]
y_train_reaction = y_train_reaction[train_valid_indices]
y_test_joke = y_test_joke[test_valid_indices]
y_test_reaction = y_test_reaction[test_valid_indices]

# Mixup data augmentation
def mixup(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    index = np.random.permutation(len(x))
    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_y = lam * y + (1 - lam) * y[index]
    return mixed_x, mixed_y

X_train_mixed, y_train_joke_mixed = mixup(X_train_features, to_categorical(y_train_joke))
_, y_train_reaction_mixed = mixup(X_train_features, to_categorical(y_train_reaction))

# Focal Loss
def focal_loss(gamma=2., alpha=.25):
    def focal_loss_fixed(y_true, y_pred):
        pt_1 = tf.where(tf.equal(y_true, 1), y_pred, tf.ones_like(y_pred))
        pt_0 = tf.where(tf.equal(y_true, 0), y_pred, tf.zeros_like(y_pred))
        return -tf.reduce_sum(alpha * tf.pow(1. - pt_1, gamma) * tf.math.log(pt_1)) - tf.reduce_sum((1-alpha) * tf.pow( pt_0, gamma) * tf.math.log(1. - pt_0))
    return focal_loss_fixed

# Attention mechanism
def attention_layer(inputs):
    attention = MultiHeadAttention(num_heads=4, key_dim=64)(inputs, inputs)
    add_attention = tf.keras.layers.Add()([inputs, attention])
    return LayerNormalization()(add_attention)

# Temporal Convolutional Network
def tcn_block(inputs, dilation_rate, nb_filters, kernel_size):
    x = Conv1D(filters=nb_filters, kernel_size=kernel_size, dilation_rate=dilation_rate, padding='causal')(inputs)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dropout(0.1)(x)
    return x

# Build model
input_shape = X_train_features.shape[1:]
num_classes_joke = len(np.unique(y_train_joke))  # Ensure this is correct
num_classes_reaction = len(np.unique(y_train_reaction))  # Ensure this is correct

inputs = Input(shape=input_shape)
x = attention_layer(inputs)

for i in range(4):
    x = tcn_block(x, dilation_rate=2**i, nb_filters=64, kernel_size=3)

x = LSTM(LSTM_UNITS, return_sequences=True)(x)
x = attention_layer(x)
x = LSTM(LSTM_UNITS // 2)(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

# Ensure the output layers match the number of classes
output_joke = Dense(num_classes_joke, activation='softmax', name='joke_output')(x)
output_reaction = Dense(num_classes_reaction, activation='softmax', name='reaction_output')(x)

model = Model(inputs=inputs, outputs=[output_joke, output_reaction])
model.compile(optimizer=Adam(), loss=[focal_loss(), focal_loss()], metrics=['accuracy', 'accuracy'])

# Train model with Monte Carlo Dropout for uncertainty estimation
train_loss = []
val_loss = []

for _ in range(MC_ITERATIONS):
    history = model.fit(
        X_train_mixed, 
        [y_train_joke_mixed, y_train_reaction_mixed], 
        epochs=EPOCHS // MC_ITERATIONS, 
        batch_size=BATCH_SIZE, 
        validation_split=0.2,
        verbose=1
    )
    train_loss.append(history.history['loss'])
    val_loss.append(history.history['val_loss'])

# Evaluate model
y_pred_joke, y_pred_reaction = model.predict(X_test_features)
y_pred_joke = np.argmax(y_pred_joke, axis=1)
y_pred_reaction = np.argmax(y_pred_reaction, axis=1)

# Print classification reports
print("Joke Type Classification Report:")
print(classification_report(y_test_joke, y_pred_joke, target_names=le_joke.classes_))
print("\nReaction Type Classification Report:")
print(classification_report(y_test_reaction, y_pred_reaction, target_names=le_reaction.classes_))

# Plot confusion matrices
plt.figure(figsize=(12, 5))
plt.subplot(121)
sns.heatmap(confusion_matrix(y_test_joke, y_pred_joke), annot=True, fmt='d', xticklabels=le_joke.classes_, yticklabels=le_joke.classes_)
plt.title('Confusion Matrix - Joke Type')
plt.subplot(122)
sns.heatmap(confusion_matrix(y_test_reaction, y_pred_reaction), annot=True, fmt='d', xticklabels=le_reaction.classes_, yticklabels=le_reaction.classes_)
plt.title('Confusion Matrix - Reaction Type')
plt.tight_layout()
plt.show()

# Plot training history with uncertainty
plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.plot(np.mean(train_loss, axis=0), label='Train')
plt.fill_between(range(EPOCHS // MC_ITERATIONS), np.mean(train_loss, axis=0) - np.std(train_loss, axis=0), 
                 np.mean(train_loss, axis=0) + np.std(train_loss, axis=0), alpha=0.2)
plt.plot(np.mean(val_loss, axis=0), label='Validation')
plt.fill_between(range(EPOCHS // MC_ITERATIONS), np.mean(val_loss, axis=0) - np.std(val_loss, axis=0), 
                 np.mean(val_loss, axis=0) + np.std(val_loss, axis=0), alpha=0.2)
plt.title('Loss with Uncertainty')
plt.legend()
plt.tight_layout()
plt.show()

# Interpretability using Integrated Gradients
def interpret_prediction(model, input_data, target_class, n_steps=50):
    baseline = np.zeros_like(input_data)
    alphas = np.linspace(0, 1, n_steps)
    grad_accumulator = np.zeros_like(input_data)
    
    for alpha in alphas:
        interpolated_input = baseline + alpha * (input_data - baseline)
        with tf.GradientTape() as tape:
            tape.watch(interpolated_input)
            predictions = model(interpolated_input)
            loss = categorical_crossentropy(target_class, predictions[0])  # Use the first output for joke type
        gradients = tape.gradient(loss, interpolated_input)
        grad_accumulator += gradients
    
    integrated_gradients = (input_data - baseline) * grad_accumulator / n_steps
    return integrated_gradients

# Interpret a sample prediction
sample_input = X_test_features[0:1]
sample_joke_class = to_categorical(y_test_joke[0], num_classes_joke)
sample_reaction_class = to_categorical(y_test_reaction[0], num_classes_reaction)

joke_interpretation = interpret_prediction(model, sample_input, sample_joke_class)
reaction_interpretation = interpret_prediction(model, sample_input, sample_reaction_class)

plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.imshow(np.mean(np.abs(joke_interpretation), axis=-1).squeeze(), cmap='viridis')
plt.title('Joke Type Interpretation')
plt.colorbar()
plt.subplot(122)
plt.imshow(np.mean(np.abs(reaction_interpretation), axis=-1).squeeze(), cmap='viridis')
plt.title('Reaction Type Interpretation')
plt.colorbar()
plt.tight_layout()
plt.show()

# Analyze feature importance
joke_importance = np.mean(np.abs(joke_interpretation), axis=(0, 1))
reaction_importance = np.mean(np.abs(reaction_interpretation), axis=(0, 1))

plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.bar(range(len(joke_importance)), joke_importance)
plt.title('Feature Importance - Joke Type')
plt.xlabel('Feature Index')
plt.ylabel('Importance')
plt.subplot(122)
plt.bar(range(len(reaction_importance)), reaction_importance)
plt.title('Feature Importance - Reaction Type')
plt.xlabel('Feature Index')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

# Print top 5 important features for each task
print("Top 5 Important Features for Joke Type Classification:")
for idx in joke_importance.argsort()[-5:][::-1]:
    print(f"Feature {idx}: {joke_importance[idx]}")

print("\nTop 5 Important Features for Reaction Type Classification:")
for idx in reaction_importance.argsort()[-5:][::-1]:
    print(f"Feature {idx}: {reaction_importance[idx]}")

# Analyze temporal importance
temporal_importance_joke = np.mean(np.abs(joke_interpretation), axis=(0, 2))
temporal_importance_reaction = np.mean(np.abs(reaction_interpretation), axis=(0, 2))

plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.plot(temporal_importance_joke)
plt.title('Temporal Importance - Joke Type')
plt.xlabel('Frame')
plt.ylabel('Importance')
plt.subplot(122)
plt.plot(temporal_importance_reaction)
plt.title('Temporal Importance - Reaction Type')
plt.xlabel('Frame')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

# Print top 5 important frames for each task
print("Top 5 Important Frames for Joke Type Classification:")
for idx in temporal_importance_joke.argsort()[-5:][::-1]:
    print(f"Frame {idx}: {temporal_importance_joke[idx]}")

print("\nTop 5 Important Frames for Reaction Type Classification:")
for idx in temporal_importance_reaction.argsort()[-5:][::-1]:
    print(f"Frame {idx}: {temporal_importance_reaction[idx]}")

Video file not found: data/videos/UGQ5W4.joke2.rep3.take1.ThatsNotAHat.mp4
Video file not found: data/videos/AQBE7J.joke1.rep3.take1.Peekaboo.mp4
Video file not found: data/videos/ZCH3UU.joke1.rep1.take1.TearingPaper.mp4
Video file not found: data/videos/C354X4.joke3.rep2.take1.TearingPaper.mp4
Video file not found: data/videos/6c6MZQ.joke1.rep2.take1.ThatsNotAHat.mp4
Video file not found: data/videos/c2KURU.joke1.rep1.take1.ThatsNotAHat.mp4
Video file not found: data/videos/SSd6Hd.joke5.rep1.take1.ThatsNotAHat.mp4
Video file not found: data/videos/KZYZFd.joke1.rep2.take1.ThatsNotACat.mp4
Video file not found: data/videos/Y3cMcU.joke4.rep3.take1.ThatsNotAHat.mp4
Video file not found: data/videos/VVDZXK.joke1.rep1.take1.ThatsNotAHat.mp4
Video file not found: data/videos/TU4BQR.joke2.rep2.take1.Peekaboo.mp4
Video file not found: data/videos/SVUcN3.joke5.rep1.take1.NomNomNom.mp4
Video file not found: data/videos/c2KURU.joke5.rep3.take1.Peekaboo.mp4
Video file not found: data/videos/A4J4QA

ValueError: No valid features extracted from any videos